## 검색 Tool 준비
- 검색 기능을 구현하지 않아도 됨

In [4]:
internet_search = {"google_search": {}}

## Deep Agent 생성
- 생성 시에 바로 
```txt
model → 사용할 LLM
tools → Agent가 사용할 Tool
system_prompt → Agent에게 역할과 행동 방식을 지시
```
설정
```txt
create_deep_agent()
       ↓
  LLM + Tools
       ↓
   Deep Agent
```

In [5]:
from deepagents import create_deep_agent
from dotenv import load_dotenv

load_dotenv()

# System prompt to steer the agent to be an expert researcher
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

agent = create_deep_agent(
    model="google_genai:gemini-3.6-flash",
    tools=[internet_search],
    system_prompt=research_instructions,
)

## 실행

In [7]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is langgraph?"}
    ]
})

print(result["messages"][-1].content)

Key 'additional_properties' is not supported in schema, ignoring
Key 'defs' is not supported in schema, ignoring
Key 'ref' is not supported in schema, ignoring
Key 'any_of' is not supported in schema, ignoring
Key 'example' is not supported in schema, ignoring
Key 'max_items' is not supported in schema, ignoring
Key 'max_length' is not supported in schema, ignoring
Key 'max_properties' is not supported in schema, ignoring
Key 'min_items' is not supported in schema, ignoring
Key 'min_length' is not supported in schema, ignoring
Key 'min_properties' is not supported in schema, ignoring
Key 'property_ordering' is not supported in schema, ignoring
Key 'additional_properties' is not supported in schema, ignoring
Key 'defs' is not supported in schema, ignoring
Key 'ref' is not supported in schema, ignoring
Key 'any_of' is not supported in schema, ignoring
Key 'example' is not supported in schema, ignoring
Key 'max_items' is not supported in schema, ignoring
Key 'max_length' is not supported 

GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

### deep agent가 자동으로 하는 일
```txt
사용자 질문
    ↓
Deep Agent
    ↓
① 검색 Tool 호출
    ↓
② 검색 결과가 크면 파일에 저장
    ↓
③ 필요하면 Subagent 생성
    ↓
④ 결과 종합
    ↓
최종 답변
```

In [8]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="google_genai:gemini-3.6-flash",
    system_prompt="Answer the user's question clearly and concisely.",
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is LangGraph?"}
    ]
})

print(result["messages"][-1].content)

[{'type': 'text', 'text': '**LangGraph** is an open-source Python and JavaScript/TypeScript framework developed by LangChain designed for building **stateful, multi-actor, and agentic LLM applications** using graph-based architectures.\n\n---\n\n### Core Concepts\n\n1. **Graph Architecture**:\n   * **Nodes**: Represent steps, functions, or agent operations (e.g., querying an LLM, calling a tool, updating a database).\n   * **Edges**: Define the control flow between nodes, including **conditional edges** for dynamic decision-making.\n\n2. **Cycles and Loops**:\n   * Standard chains and DAGs (Directed Acyclic Graphs) only move in one direction. LangGraph supports **cyclic execution**, allowing agents to loop back for multi-turn reasoning, tool execution retries, self-correction, or reflection.\n\n3. **State Management**:\n   * Maintains a shared, typed **State** object across graph executions. Each node receives the current state and returns updates to it.\n\n4. **Persistence & Human-in-

검색기능이 잘 안되는 것 같아 빼고 돌려봤다. Deep Agent가 정상적으로 여러 단계를 수행하다가 Gemini quota를 소진하는 것으로 추정된다.